In [1]:
import numpy as np 
from pathlib import Path

from ase.io import read, write
from ase import Atoms, units

[AENET](http://ann.atomistic.net/documentation/#structural-energy-reference-data) uses a specific XSF file format, that contains the total energy in its heading:  



```xsf
# total energy = -4990.44928342 eV

CRYSTAL
PRIMVEC
   2.967  0.000  0.000
   0.000  4.648  0.000
   0.000 -0.000  4.648
PRIMCOORD
6 1
Ti 1.483  2.324  2.324  0.000  0.000  0.000
Ti 0.000  0.000  0.000  0.000  0.000  0.000
O  1.483  0.905  0.905  0.000 -0.004 -0.004
O  1.483  3.742  3.742  0.000  0.004  0.004
O  0.000  1.418  3.230  0.000  0.004 -0.004
O  0.000  3.230  1.418  0.000 -0.004  0.004
```

In [2]:
# ZnO_example = read('ZnO-3.14-1.55-212-0.06-2.out', format='espresso-out')
# print(ZnO_example)
# print(f'Energia em Ry:\n{ZnO_example.get_potential_energy() / units.Ry}')
# # ZnO_example.write('ZnO-3.14-1.55-212-0.06-2.xsf', format='xsf')
# # ZnO_example
# # path_out_QE =  'ZnO-3.14-1.55-212-0.06-2.out'
# # Path(path_out_QE).with_suffix('.xsf')
# # ZnO_example

In [3]:
def qe2xsf_aenet(scf_output: str | Path, write: bool = False) -> str:
    scf_output = Path(scf_output)
    atoms: Atoms = read(scf_output, format='espresso-out')
    
    energy = atoms.get_potential_energy()
    forces = atoms.get_forces()
    
    xsf_filename = Path(scf_output).with_suffix('.xsf') # .out -> .xsf

    xsf: list[str] = ['# total energy = {} eV'.format(energy), '']
    
    if True in atoms.pbc:
        # CRYSTAL and PRIMVEC sections
        xsf += ['CRYSTAL', 'PRIMVEC']
        
        # Unit Cell Vectors
        for v in atoms.get_cell():
            xsf += ['{} {} {}'.format(*v)]

        # Atomic Positions and Forces Header
        xsf += ['PRIMCOORD', '{} 1'.format(len(atoms))]

    else:
        xsf += ['ATOMS']
    
    atom_line = ('{atom.symbol:<3s} {atom.x: .12f} {atom.y: .12f} {atom.z: .12f}'
         ' {f[0]: .12f} {f[1]: .12f} {f[2]: .12f}')
    # Append Atomic Positions and Forces after Header
    xsf += [atom_line.format(atom=atom, f=forces[i]) for i, atom in enumerate(atoms)]
    # Concatenate all lines.
    output: str = '\n'.join(xsf)

    # Write the ASE XSF file:
    if write:
        with open(xsf_filename, 'w') as f:
            f.write(output)
    
    
    return output

In [4]:
# qe2xsf_aenet('ZnO-3.14-1.55-212-0.06-2.out', write=True)
# qe2xsf_aenet('ZnO-3.14-1.55-212-0.06-2.out', write=False)


<h3>Generating ænet XSF file format </h3>

In [8]:
base_dir = Path.cwd()

xsf_dir = base_dir / "XSF_structures"
xsf_dir.mkdir(exist_ok=True)
print(f"XSF files will be saved to: {xsf_dir}")

crystalline_structures_path = Path("anisotropic_strain-Aug22")
perturbed_structures_path = Path("anisotropic_strain_pertubed")

crystalline_structures_folders: list[Path] = sorted([d for d in crystalline_structures_path.iterdir() if d.suffix == ".out"])
perturbed_structures_folders: list[Path] = sorted([d for d in perturbed_structures_path.iterdir() if d.suffix == ".out"])

all_out_files : list[Path] = []

for structure_dir in [crystalline_structures_folders, perturbed_structures_folders]:
    for direc in structure_dir:
        for file_path in sorted(f for f in direc.iterdir() if f.name.endswith(".out")):
            all_out_files.append(file_path)
        

for i, file in enumerate(all_out_files, 1):
    try:
        xsf_content: str = qe2xsf_aenet(file)
    except:
        print(f"Error in {file.name}. Probably not converged!")
       
    
    if xsf_content:
        structure_name = f"structure{i:04d}.xsf"
        
        output_path = xsf_dir / structure_name

        try:
            with open(output_path, 'w') as f:
                f.write(xsf_content)
            print(f'XSF structure of {file.name} saved => {structure_name} at {output_path}')
        except:
            print(f'Error on file {output_path}')
    else:
        print(f"No xsf content")

XSF files will be saved to: /home/jvc/QEspresso7.2/ZnO_database/scripts/XSF_structures
XSF structure of ZnO-2.94-1.45-111.out saved => structure0001.xsf at /home/jvc/QEspresso7.2/ZnO_database/scripts/XSF_structures/structure0001.xsf
XSF structure of ZnO-2.94-1.49-111.out saved => structure0002.xsf at /home/jvc/QEspresso7.2/ZnO_database/scripts/XSF_structures/structure0002.xsf
XSF structure of ZnO-2.94-1.52-111.out saved => structure0003.xsf at /home/jvc/QEspresso7.2/ZnO_database/scripts/XSF_structures/structure0003.xsf
XSF structure of ZnO-2.94-1.55-111.out saved => structure0004.xsf at /home/jvc/QEspresso7.2/ZnO_database/scripts/XSF_structures/structure0004.xsf
XSF structure of ZnO-2.94-1.58-111.out saved => structure0005.xsf at /home/jvc/QEspresso7.2/ZnO_database/scripts/XSF_structures/structure0005.xsf
XSF structure of ZnO-2.94-1.61-111.out saved => structure0006.xsf at /home/jvc/QEspresso7.2/ZnO_database/scripts/XSF_structures/structure0006.xsf
XSF structure of ZnO-2.94-1.65-111.ou